# Safety Jacket Detection - Auto-Annotation & Training

**Runs locally on a dedicated GPU.**

**Dataset:** `UCUResearchLab/Reflector-Jackets` — 301 unannotated images from HuggingFace.

**Pipeline:**
1. Install dependencies
2. Download & explore the dataset
3. Save images and create an 80/20 train/val split
4. Auto-annotate bounding boxes with **Grounding DINO**
5. Train a **YOLOv8** object detection model
6. Evaluate and view results

## Step 1: Install Dependencies

In [ ]:
!pip install -q datasets transformers ultralytics torch torchvision pillow pyyaml
print("All dependencies installed!")

## Step 2: Download & Explore the Dataset

The dataset has **301 images** in a single `train` split.

In [ ]:
import os
import random
from datasets import load_dataset
from PIL import Image

DATASET_NAME = "UCUResearchLab/Reflector-Jackets"
print(f"Downloading '{DATASET_NAME}' from HuggingFace...")

ds = load_dataset(DATASET_NAME)
print(f"\nDataset loaded: {ds}")
print(f"Features: {ds['train'].features}")
print(f"First item keys: {list(ds['train'][0].keys())}")

# --- Auto-detect the image column ---
sample = ds['train'][0]
image_col = None
for key, val in sample.items():
    if hasattr(val, 'mode') and hasattr(val, 'save'):  # PIL Image check
        image_col = key
        break

if image_col is None:
    raise ValueError("Could not find an image column. Inspect the features printed above.")

print(f"\nImage column detected: '{image_col}'")
print(f"Total images: {len(ds['train'])}")

## Step 3: Save Images to Disk & Create Train/Val Split

Splits 301 images: **80% train (~240), 20% val (~61)**.

In [ ]:
for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

all_indices = list(range(len(ds['train'])))
random.seed(42)
random.shuffle(all_indices)

split_point = int(len(all_indices) * 0.8)
train_indices = set(all_indices[:split_point])
val_indices   = set(all_indices[split_point:])

print(f"Train: {len(train_indices)} images | Val: {len(val_indices)} images")

for idx, item in enumerate(ds['train']):
    split_name = "train" if idx in train_indices else "val"
    image = item[image_col]
    if image.mode != "RGB":
        image = image.convert("RGB")
    image.save(f"dataset/images/{split_name}/img_{idx}.jpg")

print("All images saved to disk!")

## Step 4: Auto-Annotate with Grounding DINO

Uses text prompts to find reflector/safety jackets in each image and saves bounding boxes in YOLO format.

> ⚠️ This is the slowest step — expect ~10-30 min depending on your GPU.

In [ ]:
import glob
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading Grounding DINO (tiny)...")
dino = pipeline(
    task="zero-shot-object-detection",
    model="IDEA-Research/grounding-dino-tiny",
    device=device
)

CANDIDATE_LABELS = ["reflector jacket", "safety vest", "high visibility jacket", "orange safety jacket"]
CONFIDENCE_THRESHOLD = 0.25

def annotate_split(split_name):
    image_paths = sorted(glob.glob(f"dataset/images/{split_name}/*.jpg"))
    annotated, empty = 0, 0

    for i, image_path in enumerate(image_paths):
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        detections = dino(image, candidate_labels=CANDIDATE_LABELS)

        base_name = os.path.basename(image_path).replace(".jpg", ".txt")
        label_path = f"dataset/labels/{split_name}/{base_name}"

        lines = []
        for det in detections:
            if det["score"] < CONFIDENCE_THRESHOLD:
                continue
            box = det["box"]
            xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
            x_center   = ((xmin + xmax) / 2) / width
            y_center   = ((ymin + ymax) / 2) / height
            box_width  = (xmax - xmin) / width
            box_height = (ymax - ymin) / height
            lines.append(f"0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}")

        with open(label_path, "w") as f:
            f.write("\n".join(lines))

        annotated += bool(lines)
        empty += not bool(lines)

        if (i + 1) % 25 == 0:
            print(f"  [{split_name}] {i+1}/{len(image_paths)} done...")

    print(f"  [{split_name}] Finished: {annotated} annotated, {empty} empty.")

print("\nAnnotating train split...")
annotate_split("train")
print("\nAnnotating val split...")
annotate_split("val")
print("\nAnnotation complete!")

## Step 5: Train YOLOv8

In [ ]:
import yaml
from ultralytics import YOLO

dataset_yaml = {
    "path": os.path.abspath("dataset"),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "safety_jacket"}
}

with open("dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print("dataset.yaml:", open("dataset.yaml").read())

model = YOLO("yolov8n.pt")

print("Starting training...")
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device=device,
    patience=20,
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/detect",
    name="safety_jacket_v1"
)
print("Training complete!")

## Step 6: Evaluate Model

Trained weights are saved at:
```
runs/detect/safety_jacket_v1/weights/best.pt
```

Copy `best.pt` to your project's `models/` directory:
```bash
cp runs/detect/safety_jacket_v1/weights/best.pt /path/to/traffic_agent/models/best_jacket.pt
```

In [ ]:
best_model_path = "runs/detect/safety_jacket_v1/weights/best.pt"

if os.path.exists(best_model_path):
    abs_path = os.path.abspath(best_model_path)
    print(f"Model saved at:\n  {abs_path}\n")

    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(data="dataset.yaml")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
else:
    print(f"Could not find {best_model_path}. Check the runs/ directory.")